### Notebook to tweek YAIB's preprocessing of the data, to fit a SSL setup
- This means not having labels


#### YAIB preprocessing pipeline

raw data -> split data (funciton) -> preprocess data (class)

In [1]:
import os
import copy
import logging

import gin
import json
import hashlib
import pandas as pd
import polars as pl
from pathlib import Path
import pickle
from timeit import default_timer as timer
from sklearn.model_selection import StratifiedKFold, KFold, StratifiedShuffleSplit, ShuffleSplit
from icu_benchmarks.data.preprocessor import Preprocessor, PandasClassificationPreprocessor, PolarsClassificationPreprocessor
from icu_benchmarks.constants import RunMode
from icu_benchmarks.run_utils import check_required_keys
from icu_benchmarks.data.constants import DataSplit as Split, DataSegment as Segment, VarType as Var


In [2]:
from icu_benchmarks.data.split_process_data import *
from icu_benchmarks.cross_validation import execute_repeated_cv  # adjust if path is different
from icu_benchmarks.run import *
import gin

# Path ti configs
#'os.chdir("/work3/s185395/YAIB/")

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


ParsedConfigFileIncludesAndImports(filename='/work3/s185395/YAIB/configs/tasks/Regression.gin', imports=[], includes=[ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Imports.gin', imports=['icu_benchmarks.data.split_process_data', 'icu_benchmarks.data.loader', 'icu_benchmarks.models.wrappers', 'icu_benchmarks.models.dl_models', 'icu_benchmarks.models.ml_models'], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/PredictionTaskVariables.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/CrossValidation.gin', imports=[], includes=[]), ParsedConfigFileIncludesAndImports(filename='configs/tasks/common/Dataloader.gin', imports=[], includes=[])])

In [3]:
# Override the outcome scaling range
gin.bind_parameter("base_regression_preprocessor.outcome_min", 0)
gin.bind_parameter("base_regression_preprocessor.outcome_max", 10)

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [35]:
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)

<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt

#### Next step will try to be to build a dataset class that 
1. takes input like the polardataset classes in YAIB
2. computes the output like the MortalityDataset does it for R's repo 
Then we make sure that the raw datastructures can be preprocessed by YAIB's existing tools and that when the preprocessed data is loaded that it can be inputted into R's implementation 

- Maybe be aware of the paired aspect R did in her work. How is class impalance being handled by YAIB? 

This is what Chat says for now, check conversation again when working 

Polars Input
  → group by stay_id
  → sort by time
  → create (T, F) arrays
  → compute masks
  → compute deltas
  → output (data, times, static, label, mask, delta) per patient


## Understanding PredictionPolarsDataset input

In [5]:
## Input, after preprocessing 

# Data dict main structure 
data.keys()
dict_keys(['train', 'val', 'test'])

# Within each split 
data['train'].keys()
dict_keys(['OUTCOME', 'FEATURES'])

# Outcome structure
print(type(data['train']['OUTCOME']))
print(data['train']['OUTCOME'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'label']
[num_samples x 3]

# Feature structure 
print(type(data['train']['FEATURES']))
print(data['train']['FEATURES'].columns)
<class 'polars.dataframe.frame.DataFrame'>
['stay_id', 'time', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt', 'MissingIndicator_k', 'MissingIndicator_lact', 'MissingIndicator_lymph', 'MissingIndicator_map', 'MissingIndicator_mch', 'MissingIndicator_mchc', 'MissingIndicator_mcv', 'MissingIndicator_methb', 'MissingIndicator_mg', 'MissingIndicator_na', 'MissingIndicator_neut', 'MissingIndicator_o2sat', 'MissingIndicator_pco2', 'MissingIndicator_ph', 'MissingIndicator_phos', 'MissingIndicator_plt', 'MissingIndicator_po2', 'MissingIndicator_ptt', 'MissingIndicator_resp', 'MissingIndicator_sbp', 'MissingIndicator_temp', 'MissingIndicator_tnt', 'MissingIndicator_urine', 'MissingIndicator_wbc']
[num_samples x 98]

SyntaxError: invalid syntax (4206911709.py, line 14)

In [23]:
data['train']['FEATURES'].columns

['stay_id',
 'time',
 'alb',
 'alp',
 'alt',
 'ast',
 'be',
 'bicar',
 'bili',
 'bili_dir',
 'bnd',
 'bun',
 'ca',
 'cai',
 'ck',
 'ckmb',
 'cl',
 'crea',
 'crp',
 'dbp',
 'fgn',
 'fio2',
 'glu',
 'hgb',
 'hr',
 'inr_pt',
 'k',
 'lact',
 'lymph',
 'map',
 'mch',
 'mchc',
 'mcv',
 'methb',
 'mg',
 'na',
 'neut',
 'o2sat',
 'pco2',
 'ph',
 'phos',
 'plt',
 'po2',
 'ptt',
 'resp',
 'sbp',
 'temp',
 'tnt',
 'urine',
 'wbc',
 'MissingIndicator_alb',
 'MissingIndicator_alp',
 'MissingIndicator_alt',
 'MissingIndicator_ast',
 'MissingIndicator_be',
 'MissingIndicator_bicar',
 'MissingIndicator_bili',
 'MissingIndicator_bili_dir',
 'MissingIndicator_bnd',
 'MissingIndicator_bun',
 'MissingIndicator_ca',
 'MissingIndicator_cai',
 'MissingIndicator_ck',
 'MissingIndicator_ckmb',
 'MissingIndicator_cl',
 'MissingIndicator_crea',
 'MissingIndicator_crp',
 'MissingIndicator_dbp',
 'MissingIndicator_fgn',
 'MissingIndicator_fio2',
 'MissingIndicator_glu',
 'MissingIndicator_hgb',
 'MissingIndicator_

In [24]:
i = 'height'
k = True
data['train']['FEATURES'].filter(pl.col(f'MissingIndicator_{i}')==k)[[f'MissingIndicator_{i}', f'{i}']]

MissingIndicator_height,height
bool,f64
true,0.0
true,0.0
true,0.0
true,0.0
true,0.0
…,…
true,0.0
true,0.0
true,0.0


In [25]:
data['train']['FEATURES'].filter(pl.col('MissingIndicator_sex')==True)[['sex', 'MissingIndicator_sex']]

sex,MissingIndicator_sex
i64,bool


In [26]:
data['train']['FEATURES'].filter(pl.col('stay_id') == 201006).sort('time')

stay_id,time,alb,alp,alt,ast,be,bicar,bili,bili_dir,bnd,bun,ca,cai,ck,ckmb,cl,crea,crp,dbp,fgn,fio2,glu,hgb,hr,inr_pt,k,lact,lymph,map,mch,mchc,mcv,methb,mg,na,neut,…,MissingIndicator_fio2,MissingIndicator_glu,MissingIndicator_hgb,MissingIndicator_hr,MissingIndicator_inr_pt,MissingIndicator_k,MissingIndicator_lact,MissingIndicator_lymph,MissingIndicator_map,MissingIndicator_mch,MissingIndicator_mchc,MissingIndicator_mcv,MissingIndicator_methb,MissingIndicator_mg,MissingIndicator_na,MissingIndicator_neut,MissingIndicator_o2sat,MissingIndicator_pco2,MissingIndicator_ph,MissingIndicator_phos,MissingIndicator_plt,MissingIndicator_po2,MissingIndicator_ptt,MissingIndicator_resp,MissingIndicator_sbp,MissingIndicator_temp,MissingIndicator_tnt,MissingIndicator_urine,MissingIndicator_wbc,age,sex,height,weight,MissingIndicator_age,MissingIndicator_sex,MissingIndicator_height,MissingIndicator_weight
i64,duration[ms],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,i64,f64,f64,bool,bool,bool,bool
201006,0ms,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.516011,0.0,0.0,0.454634,-0.930323,0.681749,-0.580126,-0.658271,0.0,-0.61205,0.422837,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,false,false,false,false,false,true,false,false,false,false,false,true,false,false,false,false,true,true,false,false,true,false,false,false,true,true,true,false,-0.16248,1,-0.943257,-0.888026,false,false,false,false
201006,1h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.619331,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.735582,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true,-0.16248,1,-0.943257,-0.888026,false,false,false,false
201006,2h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.791532,0.0,0.0,0.454634,-0.930323,0.925322,-0.580126,-0.658271,0.0,-0.61205,0.90813,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,false,true,-0.16248,1,-0.943257,-0.888026,false,false,false,false
201006,3h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.860412,0.0,0.0,0.454634,-0.930323,0.844131,-0.580126,-0.658271,0.0,-0.61205,1.199306,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,false,true,true,true,-0.16248,1,-0.943257,-0.888026,false,false,false,false
201006,4h,-0.60284,0.0,0.0,0.0,0.0,-0.338794,0.0,0.0,-0.671308,-0.640933,-0.865833,0.0,0.0,0.0,-0.69228,-0.488212,0.0,0.17161,0.0,0.0,0.454634,-0.930323,0.140477,-0.580126,-0.658271,0.0,-0.61205,0.034602,-0.763181,-0.987222,-0.101033,0.0,-0.750164,-1.599037,0.737334,…,true,true,true,false,true,true,true,true,false,true,true,true,true,true,true,true,false,true,true,true,true,true,true,false,false,true,true,true,true,-0.16248,1,-0.943257,-0.888026,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
201006,6d 20h,-0.60284,0.0,0.0,0.0,-1.290317,0.221696,0.0,0.0,-0.041957,1.679445,1.03133,2.202911,0.0,0.0,-0.69228,1.108013,0.0,1.067052,0.0,2.424732,-0.639334,0.761649,1.331277,-0.580126,2.467983,-0.758467,-1.038713,

## Understanding PredictionPolarsDataset output 

In [38]:
type(data)

dict

In [43]:
from icu_benchmarks.data.loader import PredictionPolarsDataset, CommonPolarsDataset
from torch.utils.data import DataLoader

dataset_class = PredictionPolarsDataset
#dataset_class = CommonPolarsDataset

batch_size=1

train_dataset = dataset_class(data, split=Split.train, ram_cache=False)
train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
    )

# Get one batch from the DataLoader
for batch_idx, (data, labels, pad_mask) in enumerate(train_loader):
    # Print the data and other details of the batch
    print(f"Batch {batch_idx + 1}:")
    print("Data (tensor):", data)

    # Break after the first batch to just inspect one batch
    break

['stay_id', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt', 'MissingIndicator_k', 'MissingIndicator_lact', '

In [11]:
# Get one batch from the DataLoader
for batch_idx, (data, labels, pad_mask) in enumerate(train_loader):
    # Print the data and other details of the batch
    print(f"Batch {batch_idx + 1}:")
    print("Data (tensor):", data)
    print("Labels (tensor):", labels)
    print("Padding Mask (tensor):", pad_mask)
    
    # Optionally, inspect the shape of the batch tensors
    print(f"Data shape: {data.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Padding Mask shape: {pad_mask.shape}")
    
    # Break after the first batch to just inspect one batch
    break


['stay_id', 'alb', 'alp', 'alt', 'ast', 'be', 'bicar', 'bili', 'bili_dir', 'bnd', 'bun', 'ca', 'cai', 'ck', 'ckmb', 'cl', 'crea', 'crp', 'dbp', 'fgn', 'fio2', 'glu', 'hgb', 'hr', 'inr_pt', 'k', 'lact', 'lymph', 'map', 'mch', 'mchc', 'mcv', 'methb', 'mg', 'na', 'neut', 'o2sat', 'pco2', 'ph', 'phos', 'plt', 'po2', 'ptt', 'resp', 'sbp', 'temp', 'tnt', 'urine', 'wbc', 'MissingIndicator_alb', 'MissingIndicator_alp', 'MissingIndicator_alt', 'MissingIndicator_ast', 'MissingIndicator_be', 'MissingIndicator_bicar', 'MissingIndicator_bili', 'MissingIndicator_bili_dir', 'MissingIndicator_bnd', 'MissingIndicator_bun', 'MissingIndicator_ca', 'MissingIndicator_cai', 'MissingIndicator_ck', 'MissingIndicator_ckmb', 'MissingIndicator_cl', 'MissingIndicator_crea', 'MissingIndicator_crp', 'MissingIndicator_dbp', 'MissingIndicator_fgn', 'MissingIndicator_fio2', 'MissingIndicator_glu', 'MissingIndicator_hgb', 'MissingIndicator_hr', 'MissingIndicator_inr_pt', 'MissingIndicator_k', 'MissingIndicator_lact', '

#### Mortality Dataset output

In [ ]:
# I need to create all of these 6 from the preprocessed data format above 

Dynamic Features 
[batch_size, num_sensors, num_timesteps]

Static Features
[batch_size, num_static_features]

label_array
[batch_size, 1]

Sensor Mask
[batch_size, num_sensors, num_timesteps]

Time Features
[batch_size, num_timesteps]

Delta Features
[batch_size, num_sensors, num_timesteps]

### Version 2

In [ ]:
# Second version, includes observation and forecasting + masks 
# doesnot handle static, delta, original mask yet 

import torch
import numpy as np
import polars as pl
from torch.utils.data import Dataset
from typing import Dict, Tuple 



class SSLPolarsDataset(CommonPolarsDataset):
    """Subclass of common dataset for prediction tasks.

    Args:
        ram_cache (bool, optional): Whether the complete dataset should be stored in ram. Defaults to True.
    """

    def __init__(self, *args, ram_cache: bool = True, **kwargs):
        super().__init__(*args, **kwargs)
        self.ram_cache(ram_cache)
    
    def select_observation_forecasting_windows(self, window: np.ndarray, missingness_mask: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Method to randomly select the observation window (t0 to t1) and the corresponding forecasting window (t2),
        along with the corresponding observation and forecasting masks.

        Args:
            window: The data for the current stay_id (time series).
            missingness_mask: The missingness mask for the data (1 for observed, 0 for missing).

        Returns:
            A tuple (obs_window, forecast_window, obs_mask, forecast_mask), where all are numpy arrays.
        """
        valid_t2_found = False
        forecast_window = None
        obs_mask = None
        forecast_mask = None

        # Try sampling t1 until we find a valid t2
        while not valid_t2_found:
            # Randomly select t1 (forecasting time) from the available timestamps
            t1_ix = np.random.choice(len(window))  # t1 can be any index within the available time steps

            # Define t0 as the start of the observation window, ensuring it doesn't go below 0
            t0_ix = max(0, t1_ix - self.max_obs)

            # Slice the observation window
            obs_window = window[t0_ix:t1_ix]  # Observation window (from t0 to t1)
            obs_mask = missingness_mask[t0_ix:t1_ix]  # Corresponding observation mask (same length as obs_window)

            # Search for the first time step after t1 with at least one observed value in that time step
            for t2_ix in range(t1_ix, len(window)):
                # Check if at least one value in this time step is observed (based on missingness mask)
                if np.any(missingness_mask[t2_ix] == 1):  # 1 means observed, 0 means missing
                    forecast_window = window[t2_ix:t2_ix + 1]  # Forecast window (just one time step after t1)
                    forecast_mask = missingness_mask[t2_ix:t2_ix + 1]  # Forecasting mask (same length as forecast_window)
                    valid_t2_found = True  # Mark that we found a valid forecasting window
                    break  # Exit the loop once a valid t2 is found

            if not valid_t2_found:
                # If no valid t2 is found, randomly sample another t1 and try again
                continue

        return obs_window, forecast_window, obs_mask, forecast_mask

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Function to sample from the data split of choice. Used for deep learning implementations.

        Args:
            idx: A specific row index to sample.

        Returns:
            A sample from the data, consisting of data, labels, padding mask, and other arrays.
        """
        if self._cached_dataset is not None:
            return self._cached_dataset[idx]
        
        # Extracting the stay_id for the specific index
        stay_id = self.outcome_df[self.vars["GROUP"]].unique()[idx] 
    
        # Select dynamic values (excluding stay_id and time columns)
        dynamic_columns = self.vars["DYNAMIC"]  # Use DYNAMIC columns defined in gin
        data = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(dynamic_columns).to_numpy()
        
        # Select missingness indicators for dynamic features (matching the same order as the dynamic features)
        missingness_columns = [f'MissingIndicator_{col}' for col in dynamic_columns]
        data_mask = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(missingness_columns).to_numpy()
        data_mask = (1 - data_mask).to_numpy()

        # Select static features (assuming they are labeled 'age', 'sex', etc. in the dataset)
        static_columns = self.vars["STATIC"]  # Use STATIC columns defined in gin
        static = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(static_columns).to_numpy()
        
        # Select missingness indicators for static features
        #static_missingness_columns = [f'MissingIndicator_{col}' for col in static_columns]
        #static_mask = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(static_missingness_columns).to_numpy()
        #static_mask = (1 - static_mask).to_numpy()

        # Array containing times values call it times 
        time_column = self.vars["SEQUENCE"]
        times = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(time_column).to_numpy()
        times = np.array([t.total_seconds() / 60 for t in times])  # Convert timedelta to minutes

        # Array containing delta values 
        delta = self.get_delta_t(times, data, data_mask)

        # Padding the data and corresponding masks to match max_length
        length_diff = self.max_length - data.shape[0]
        if length_diff > 0:
            data = np.pad(data, ((0, length_diff), (0, 0)), mode='constant', constant_values=0)
            data_mask = np.pad(data_mask, ((0, length_diff), (0, 0)), mode='constant', constant_values=0)

        # Get the observation and forecasting windows and their corresponding masks
        obs_window, forecast_window, obs_mask, forecast_mask = self.select_observation_forecasting_windows(data, data_mask)

        # Return all of the required arrays as tensors
        return (
            torch.from_numpy(obs_window).float(),
            torch.from_numpy(forecast_window).float(),
            torch.from_numpy(obs_mask).bool(),
            torch.from_numpy(forecast_mask).bool(),
            torch.from_numpy(times), 
            torch.from_numpy(static),  
            torch.from_numpy(delta)
        )

    def get_delta_t(times, measurements, measurement_indicators):
        """
        From R's repo 
        Creates array with time difference from the most recent feature measurement.
        """
        dt_list = []

        # First observation has delta t = 0
        first_dt = np.zeros(measurement_indicators.shape[1:], dtype=np.float32)  # (F,)
        dt_list.append(first_dt)

        last_dt = first_dt.copy()  # Initialize last_dt before the loop
        for i in range(1, measurement_indicators.shape[0]):
            # Calculate time difference only for observed values
            last_dt = np.where(
                measurement_indicators[i - 1],  # If the previous value was observed
                np.full_like(last_dt, times[i] - times[i - 1]),  # Compute time difference
                times[i] - times[i - 1] + last_dt,  # If the previous value was missing, propagate the last valid time difference
            )
            dt_list.append(last_dt)

        dt_array = np.stack(dt_list)  # Combine the list of deltas into a single array
        dt_array = dt_array.astype(np.float32)  # Ensure consistent data type
        dt_array.shape = measurements.shape  # Reshape to match measurements
        dt_array = dt_array * ~(measurement_indicators.astype(bool))  # Mask the missing values

        return dt_array

    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.
        """
        return len(self.outcome_df[self.vars["GROUP"]])


SyntaxError: invalid syntax (1952104181.py, line 9)

## Debug version 2

In [3]:
vars_dict = {
    "GROUP": "stay_id",
    "SEQUENCE": "time",
    "LABEL": "label",
    "DYNAMIC": ["alb", "alp", "alt", "ast", "be", "bicar", "bili", "bili_dir", "bnd", "bun", "ca", "cai", "ck", "ckmb", "cl",
        "crea", "crp", "dbp", "fgn", "fio2", "glu", "hgb", "hr", "inr_pt", "k", "lact", "lymph", "map", "mch", "mchc", "mcv",
        "methb", "mg", "na", "neut", "o2sat", "pco2", "ph", "phos", "plt", "po2", "ptt", "resp", "sbp", "temp", "tnt", "urine",
        "wbc"],
    "STATIC": ["age", "sex", "height", "weight"],
}

In [4]:
# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [4]:
# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("/work3/s185395/YAIB/demo_data/mortality24/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.classification
)


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [5]:
ids = []
for i in range(len(data['test']['FEATURES']['stay_id'].unique())):
    ids.append(data['test']['FEATURES']['stay_id'].unique()[i])

In [6]:
lengths = []
for i in range(len(ids)):
    lengths.append(len(data['test']['OUTCOME'].filter(pl.col('stay_id') == ids[i])['time']))
   

In [7]:
lengths

[46,
 16,
 69,
 169,
 31,
 32,
 86,
 50,
 114,
 27,
 32,
 42,
 169,
 123,
 24,
 169,
 22,
 91,
 24,
 119,
 47,
 169,
 59,
 32,
 62,
 15,
 26]

# TO DO NEXT try the new class on R's model, can you set up simple training? \n first check how times are normalized in seft. does this effect the deltas too?

In [61]:
# Second version, includes observation and forecasting + masks 
# doesnot handle static, delta, original mask yet 

import torch
import numpy as np
import polars as pl
from torch.utils.data import Dataset
from typing import Dict, Tuple
from icu_benchmarks.data.loader import CommonPolarsDataset
from torch.utils.data import DataLoader
from icu_benchmarks.constants import RunMode
from torch.nn.functional import pad

class BATPolarsDataset(CommonPolarsDataset):
    """Subclass of common dataset for prediction tasks.

    Args:
        ram_cache (bool, optional): Whether the complete dataset should be stored in ram. Defaults to True.
    """

    def __init__(self, *args, ram_cache: bool = True, runmode=None, **kwargs):
        #super().__init__(*args, **kwargs)
        #self.outcome_df = self.grouping_df
        #self.ram_cache(ram_cache)

        # TEMP BEFORE GIN SETUP AND UPLOAD OF CLASS IN LOADER.PY 
        # Explicitly pass the vars_dict to the parent class (CommonPolarsDataset)
        self.vars = vars_dict  # Define vars_dict manually
        # Pass all arguments to the parent class, including 'vars'
        super().__init__(vars=self.vars, **kwargs)
        self.outcome_df = self.grouping_df
        self.ram_cache(ram_cache)
        self.runmode = runmode
    

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Function to sample from the data split of choice. Used for deep learning implementations.

        Args:
            idx: A specific row index to sample.

        Returns:
            A sample from the data, consisting of data, labels, padding mask, and other arrays.
        """
        if self._cached_dataset is not None:
            return self._cached_dataset[idx]

        # UNCOMMENT FOR NORMALIZATION OF TIME For normalizing times array (has not been done in preprocessing) 
        #global_max_time_ms = (
        #self.features_df.select(pl.col(self.vars["SEQUENCE"]).max()).item().total_seconds() * 1000  # Convert to milliseconds
        #)
        
        # Extracting the stay_id for the specific index
        stay_id = self.outcome_df[self.vars["GROUP"]].unique()[idx]  

        # Selecting label column 
        label = self.outcome_df.filter(pl.col(self.vars["GROUP"]) == stay_id)[self.vars["LABEL"]].to_numpy()
        
        # Select dynamic values (excluding stay_id and time columns)
        dynamic_columns = self.vars["DYNAMIC"]  # Use DYNAMIC columns defined in gin
        data = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(dynamic_columns).to_numpy()
        
        # Select missingness indicators for dynamic features (matching the same order as the dynamic features)
        missingness_columns = [f'MissingIndicator_{col}' for col in dynamic_columns]
        mask = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(missingness_columns).to_numpy()
        mask = (1 - mask)

        # Select static features (assuming they are labeled 'age', 'sex', etc. in the dataset)
        static_columns = self.vars["STATIC"]  # Use STATIC columns defined in gin
        static = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(static_columns)[0].to_numpy().flatten()

        # Select timeseries 
        time_column = self.vars["SEQUENCE"]
        times_raw = self.features_df.filter(pl.col(self.vars["GROUP"]) == stay_id).select(time_column).to_numpy().flatten()
        # tmp solution for times normalization, should be done in preprocessing. See how R did it 
        times_numeric = (times_raw - times_raw[0]).astype('timedelta64[ms]').astype(np.float32)
        # NO NORNALIZATION: Convert milliseconds to minutes
        times = times_numeric / 60000  # 1 minute = 60,000 ms
        # NORMALIZATION: to [-1, 1] based on global max in milliseconds
        #times = (times_numeric / global_max_time_ms) * 2 - 1

        # Array containing delta time values 
        delta = self.get_delta_t(times, data, mask) 

        # Permute 
        data = data.T
        mask = mask.T
        delta = delta.T
    
        # Return all of the required arrays as tensors
        return (
            torch.from_numpy(data),
            torch.from_numpy(mask),
            torch.from_numpy(label),
            torch.from_numpy(times),
            torch.from_numpy(static),
            torch.from_numpy(delta)
            )
    
    @staticmethod
    def get_delta_t(times, measurements, measurement_indicators):
        """
        From R's repo 
        Creates array with time difference from the most recent feature measurement.
        """
        dt_list = []

        # First observation has delta t = 0
        first_dt = np.zeros(measurement_indicators.shape[1:], dtype=np.float32)  # (F,)
        dt_list.append(first_dt)

        last_dt = first_dt.copy()  # Initialize last_dt before the loop
        for i in range(1, measurement_indicators.shape[0]):
            # Calculate time difference only for observed values
            last_dt = np.where(
                measurement_indicators[i - 1],  # If the previous value was observed
                np.full_like(last_dt, times[i] - times[i - 1]),  # Compute time difference
                times[i] - times[i - 1] + last_dt,  # If the previous value was missing, propagate the last valid time difference
            )
            dt_list.append(last_dt)

        dt_array = np.stack(dt_list)  # Combine the list of deltas into a single array
        dt_array = dt_array.astype(np.float32)  # Ensure consistent data type
        dt_array.shape = measurements.shape  # Reshape to match measurements
        dt_array = dt_array * ~(measurement_indicators.astype(bool))  # Mask the missing values

        return dt_array

   
    def collate_fn_pad_to_longest_in_batch(self):
        def collate_fn(batch):
            data, mask, label, times, static, delta = zip(*batch)
            max_len = max(x.shape[-1] for x in data)
            original_lengths = [x.shape[-1] for x in data]
            
            def pad_2d_tensor(tensor, max_len):
                pad_amt = max_len - tensor.shape[-1]
                return pad(tensor, (0, pad_amt)) if pad_amt > 0 else tensor
            
            def pad_1d_tensor(tensor, max_len):
                pad_amt = max_len - tensor.shape[0]
                return pad(tensor, (0, pad_amt)) if pad_amt > 0 else tensor

            data   = torch.stack([pad_2d_tensor(x, max_len) for x in data])
            mask   = torch.stack([pad_2d_tensor(x, max_len) for x in mask])
            delta  = torch.stack([pad_2d_tensor(x, max_len) for x in delta])
            times  = torch.stack([pad_1d_tensor(x, max_len) for x in times])
            static = torch.stack(static)

            # In a regression setting there is one label per time bin 
            if self.runmode == RunMode.regression:
                label = torch.stack([pad_1d_tensor(x, max_len) for x in label])
            # In a classification setting there is one label per patient/stay_id 
            else:
                label = torch.stack(label).squeeze()   

            obs_mask = torch.zeros((len(data), max_len), dtype=torch.int32)
            for i, seq_len in enumerate(original_lengths):
                obs_mask[i, :seq_len] = 1
            
            return data, mask, label, times, static, delta, obs_mask

        return collate_fn
    
    def __len__(self) -> int:
        """
        Return the total number of samples in the dataset.
        """
        return self.outcome_df[self.vars["GROUP"]].n_unique()
    


In [63]:
# Initialize your dataset class
dataset_class = BATPolarsDataset

# Initialize dataset instance
train_dataset = dataset_class(data=data, split="test", ram_cache=False, runmode = RunMode.regression)
#train_dataset = dataset_class(data=data, split="test", ram_cache=False, runmode = RunMode.classification)

# Create a DataLoader
batch_size = 3  # Adjust batch size as needed
train_loader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=train_dataset.collate_fn_pad_to_longest_in_batch())

# Step 2: Inspect one batch from the DataLoader

# Get one batch from the DataLoader
for batch_idx, (data, mask, labels, times ,static, delta, obs_mask) in enumerate(train_loader):
    #print(f"Batch {batch_idx + 1}:")
    print("Data (tensor):", data.shape)
    print("Data Mask (tensor):", mask.shape) 
    print("Labels (tensor):", labels.shape)
    print("Times (tensor):", times.shape)
    print("Static (tensor):", static.shape)
    print("Delta (tensor):", delta.shape)
    print("Observation mask (tensor):", obs_mask.shape)
    
    # Break after the first batch to inspect one batch
    break

Data (tensor): torch.Size([3, 48, 69])
Data Mask (tensor): torch.Size([3, 48, 69])
Labels (tensor): torch.Size([3, 69])
Times (tensor): torch.Size([3, 69])
Static (tensor): torch.Size([3, 4])
Delta (tensor): torch.Size([3, 48, 69])
Observation mask (tensor): torch.Size([3, 69])


In [64]:
times

tensor([[   0.,   60.,  120.,  180.,  240.,  300.,  360.,  420.,  480.,  540.,
          600.,  660.,  720.,  780.,  840.,  900.,  960., 1020., 1080., 1140.,
         1200., 1260., 1320., 1380., 1440., 1500., 1560., 1620., 1680., 1740.,
         1800., 1860., 1920., 1980., 2040., 2100., 2160., 2220., 2280., 2340.,
         2400., 2460., 2520., 2580., 2640., 2700.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.],
        [   0.,   60.,  120.,  180.,  240.,  300.,  360.,  420.,  480.,  540.,
          600.,  660.,  720.,  780.,  840.,  900.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,    0.,
            0.,    0.,    0.,    0.,    0.,    0.,    0., 

In [11]:
# RELOAD DATASET CLASSIFICATION
# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/BinaryClassification.gin")

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("/work3/s185395/YAIB/demo_data/mortality24/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.classification
)


/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


In [67]:
# RELOAD DATASET REGRESSION

# Load the gin config
gin.parse_config_file("/work3/s185395/YAIB/configs/tasks/Regression.gin")

# Call the preprocessing function with the new scale
data = preprocess_data(
    data_dir=Path("demo_data/los/mimic_demo"),
    seed=2222,
    generate_cache=True,
    load_cache=False,
    debug=False,
    use_static = True,
    runmode=RunMode.regression,
)

/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1140: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1145: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/work3/s185395/YAIB/yaib_venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:1165: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


## First version of SSL dataset that inherits from BATPolarsDataset

In [68]:
# Initialize your dataset class
dataset_class = SSLPolarsDataset

# Initialize dataset instance
ssl_dataset = dataset_class(data=data, split="test", ram_cache=False, runmode=RunMode.regression)

# Create a DataLoader using the SSL-specific collate function
batch_size = 3
ssl_loader = DataLoader(ssl_dataset, batch_size=batch_size, collate_fn=ssl_dataset.collate_fn_ssl_windows())

# Inspect one batch from the DataLoader
for batch_idx, batch in enumerate(ssl_loader):
    print("obs_data (tensor):", batch['obs_data'].shape)
    print("obs_mask (tensor):", batch['obs_mask'].shape)
    print("obs_times (tensor):", batch['obs_times'].shape)
    print("obs_delta (tensor):", batch['obs_delta'].shape)
    print("forecast_target (tensor):", batch['forecast_target'].shape)
    print("forecast_mask (tensor):", batch['forecast_mask'].shape)
    print("static (tensor):", batch['static'].shape)

    # Optional: print first observation mask
    print("First obs_mask row:", batch['obs_mask'][0])

    break  # Stop after first batch


obs_data (tensor): torch.Size([3, 48, 13])
obs_mask (tensor): torch.Size([3, 48, 13])
obs_times (tensor): torch.Size([3, 13])
obs_delta (tensor): torch.Size([3, 48, 13])
forecast_target (tensor): torch.Size([3, 48, 2])
forecast_mask (tensor): torch.Size([3, 48, 2])
static (tensor): torch.Size([3, 4])
First obs_mask row: tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [16]:
p = 0
for i in range(len(batch['obs_data'][p,:,0])):
    if not torch.all(batch['forecast_mask'][p,i,:] == 0):
        print(i)
        print('Observation window')
        print(batch['obs_data'][p,i,:])
        print(batch['obs_mask'][p,i,:])
        #print(batch['obs_delta'][p,i,:])
        print('\nForecasting window')
        print(batch['forecast_target'][p,i,:])
        print(batch['forecast_mask'][p,i,:])


17
Observation window
tensor([-0.3106, -0.3106, -1.0682, -0.1728, -0.0695, -0.7238, -0.7238, -0.5172,
        -0.4483, -0.7927,  0.5160, -0.3106, -1.0682, -0.9305, -1.1371,  0.6538,
        -0.6550, -1.2060, -0.8616, -0.9994, -0.2417,  0.0338, -1.2749, -0.8616],
       dtype=torch.float64)
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

Forecasting window
tensor([-1.0682, -0.6550], dtype=torch.float64)
tensor([1, 1])
22
Observation window
tensor([-1.1586, -0.9962, -0.8338, -1.1045, -0.6985, -0.4549, -0.6714, -0.4008,
        -0.7797, -0.8879, -1.2668, -1.2127, -1.3751, -1.5375, -1.4833, -1.1586,
        -1.0503, -0.8338, -1.1045, -0.8609, -1.0503, -0.8879, -1.1586, -1.4833],
       dtype=torch.float64)
tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])

Forecasting window
tensor([-1.5916, -1.5916], dtype=torch.float64)
tensor([1, 1])
27
Observation window
tensor([ 0.0993, -0.3536, -1.3242,  0.3581,  0.1964, -0.8713, -1.065

In [ ]:
import torch
import random
import numpy as np
#from loader import BATPolarsDataset  # Adjust import path as needed

class SSLPolarsDataset(BATPolarsDataset):
    def __init__(self, *args, max_obs=24, forecast_horizon=2, **kwargs):
        """
        SSL dataset that slices each batch into observation and forecasting windows.

        Args:
            max_obs (int): Length of the observation window in time bins (e.g., 24 = 24h).
            forecast_horizon (int): Length of the forecasting window in time bins (e.g., 2 = 2h).
        """
        super().__init__(*args, **kwargs)
        self.max_obs = max_obs
        self.forecast_horizon = forecast_horizon

    def collate_fn_ssl_windows(self):
        base_collate = super().collate_fn_pad_to_longest_in_batch()

        def collate_fn(batch):
            data, mask, label, times, static, delta, obs_mask = base_collate(batch)
            B, C, T = data.shape

            t1_ix = None
            tries = 0
            max_tries = B  # Retry up to one attempt per patient in the batch

            while t1_ix is None and tries < max_tries:
                patient_idx = random.randint(0, B - 1)
                patient_mask = obs_mask[patient_idx].bool()
                valid_indices = torch.where(patient_mask)[0]

                # Enforce: minimum 12 time bins of history and room for forecast
                valid_indices = valid_indices[valid_indices >= 12]
                valid_indices = valid_indices[valid_indices <= valid_indices[-1] - self.forecast_horizon]

                if len(valid_indices) > 0:
                    t1_ix = int(np.random.choice(valid_indices.cpu().numpy()))
                    break

                tries += 1

            if t1_ix is None:
                raise ValueError("No valid t1 index found in batch after retrying.")

            t0_ix = max(0, t1_ix - self.max_obs)
            t2_ix = t1_ix + self.forecast_horizon

            # Slice all patients at same window
            obs_data = data[:, :, t0_ix:t1_ix]
            obs_mask_out = mask[:, :, t0_ix:t1_ix]
            obs_times = times[:, t0_ix:t1_ix]
            obs_delta = delta[:, :, t0_ix:t1_ix]

            forecast_target = data[:, :, t1_ix:t2_ix]
            forecast_mask = mask[:, :, t1_ix:t2_ix]

            return {
                    'obs_data': obs_data,
                    'obs_mask': obs_mask_out,
                    'obs_times': obs_times,
                    'obs_delta': obs_delta,
                    'forecast_target': forecast_target,
                    'forecast_mask': forecast_mask,
                    'static': static,
                    #'debug': {  # UNCOMMENT FOR DEBUGGING
                    #    't0_ix': t0_ix,
                    #    't1_ix': t1_ix,
                    #    't2_ix': t2_ix
                    #    }
                    }
        return collate_fn


### Testing the BAT and SSL PolarDataset classes

In [75]:
# Initialize datasets
bat_dataset = BATPolarsDataset(data=data, split="test", ram_cache=False, runmode = RunMode.regression)
ssl_dataset = SSLPolarsDataset(data=data, split="test", ram_cache=False, runmode=RunMode.regression)

# Get a sample batch
base_batch = next(iter(DataLoader(bat_dataset, batch_size=10, collate_fn=bat_dataset.collate_fn_pad_to_longest_in_batch())))
ssl_batch = next(iter(DataLoader(ssl_dataset, batch_size=10, collate_fn=ssl_dataset.collate_fn_ssl_windows())))


In [76]:
# Compare base batch (BAT) to SSL slices

# Slicing times 
t0 = ssl_batch['debug']['t0_ix']
t1 = ssl_batch['debug']['t1_ix']
t2 = ssl_batch['debug']['t2_ix']

# Observation window
if torch.allclose(ssl_batch['obs_data'][0], base_batch[0][0, :, t0:t1]):
    print("✅ obs_data matches original observation window.")
else:
    print("❌ obs_data does NOT match.")

# Observation mask
if torch.allclose(ssl_batch['obs_mask'][0], base_batch[1][0, :, t0:t1]):
    print("✅ obs_mask matches original observation window mask.")
else:
    print("❌ obs_mask does NOT match.")

# Forecasting window
if torch.allclose(ssl_batch['forecast_target'][0], base_batch[0][0, :, t1:t2]):
    print("✅ forecast_target matches original forecasting window.")
else:
    print("❌ forecast_target does NOT match.")

# Forecasting mask
if torch.allclose(ssl_batch['forecast_mask'][0], base_batch[1][0, :, t1:t2]):
    print("✅ forecast_mask matches original forecasting window mask.")
else:
    print("❌ forecast_mask does NOT match.")

# Times for observation window
if torch.allclose(ssl_batch['obs_times'][0], base_batch[3][0, t0:t1]):
    print("✅ obs_times match original times in observation window.")
else:
    print("❌ obs_times do NOT match.")

# Delta between times in observation window
if torch.allclose(ssl_batch['obs_delta'][0], base_batch[5][0, :, t0:t1]):
    print("✅ obs_delta matches original deltas in observation window.")
else:
    print("❌ obs_delta does NOT match.")

# Static features
if torch.allclose(ssl_batch['static'][0], base_batch[4][0, :]):
    print("✅ static features match.")
else:
    print("❌ static features do NOT match.")

✅ obs_data matches original observation window.
✅ obs_mask matches original observation window mask.
✅ forecast_target matches original forecasting window.
✅ forecast_mask matches original forecasting window mask.
✅ obs_times match original times in observation window.
✅ obs_delta matches original deltas in observation window.
✅ static features match.


#### Next step will be to try and deduct the static, dynamic and the masks for dynamic array in the new dataset function 

remember to check how R handles missing static value, how does she impute them? then you should do the same 
Try and understand how the commonpolarsdataset class is workig
- are they doing some grouping and sorting?? maybe you should do the same
- How are they hadling the stay_id and the time column? i think maybe they are grouping and sorting based on this
- Continue in your chat BaseModule Class Breakdown 

### Below is the actual format of some preprocessed data that ran through their pipeline

In [39]:
import polars as pl
import os

# Path where your Parquet files are saved
folder_path = "/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data_test"

# List all the Parquet files in the folder
parquet_files = [f for f in os.listdir(folder_path) if f.endswith('.parquet')]

# Load and inspect each Parquet file
for parquet_file in parquet_files:
    file_path = os.path.join(folder_path, parquet_file)
    
    # Load the Parquet file into a Polars DataFrame
    df = pl.read_parquet(file_path)
    
    # Print the file name and inspect the first few rows of the DataFrame
    print(f"Inspecting {parquet_file}:")
    print(df.head())  # Shows the first few rows
    print(df.shape)  # Shows the number of rows and columns
    print(df.columns)  # Shows the column names
    print("\n")  # Add an empty line for readability


Inspecting train_FEATURES.parquet:
shape: (5, 106)
┌─────────┬────────────┬───────────┬──────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ stay_id ┆ time       ┆ alb       ┆ alp      ┆ … ┆ MissingInd ┆ MissingIn ┆ MissingIn ┆ MissingIn │
│ ---     ┆ ---        ┆ ---       ┆ ---      ┆   ┆ icator_age ┆ dicator_s ┆ dicator_h ┆ dicator_w │
│ i64     ┆ duration[m ┆ f64       ┆ f64      ┆   ┆ ---        ┆ ex        ┆ eight     ┆ eight     │
│         ┆ s]         ┆           ┆          ┆   ┆ bool       ┆ ---       ┆ ---       ┆ ---       │
│         ┆            ┆           ┆          ┆   ┆            ┆ bool      ┆ bool      ┆ bool      │
╞═════════╪════════════╪═══════════╪══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 223870  ┆ 2h         ┆ -1.048117 ┆ 1.319224 ┆ … ┆ false      ┆ false     ┆ false     ┆ false     │
│ 281609  ┆ 1d         ┆ 0.0       ┆ 0.0      ┆ … ┆ false      ┆ false     ┆ false     ┆ false     │
│ 283875  ┆ 2h         ┆ 0.0       ┆ 0.0

In [18]:
data = {"missing": [True, False]}
df = pl.DataFrame(data)
df = df.with_columns((~pl.col("missing")).cast(pl.Int8).alias("observed"))
df

missing,observed
bool,i8
true,0
false,1


In [51]:
import polars as pl
import os

# Path where your Parquet files are saved
folder_path = "/work3/s185395/YAIB/icu_benchmarks/data/preprocessed_data_test"

# List all the Parquet files in the folder
#parquet_file = 'train_FEATURES.parquet'
parquet_file = 'train_OUTCOME.parquet'

file_path = os.path.join(folder_path, parquet_file)

# Load the Parquet file into a Polars DataFrame
df = pl.read_parquet(file_path)
print(df.head())  # Shows the first few rows



shape: (5, 3)
┌─────────┬──────────────┬──────────┐
│ stay_id ┆ time         ┆ label    │
│ ---     ┆ ---          ┆ ---      │
│ i64     ┆ duration[ms] ┆ f64      │
╞═════════╪══════════════╪══════════╡
│ 200001  ┆ 0ms          ┆ 0.324444 │
│ 200001  ┆ 1h           ┆ 0.32     │
│ 200001  ┆ 2h           ┆ 0.315556 │
│ 200001  ┆ 3h           ┆ 0.311111 │
│ 200001  ┆ 4h           ┆ 0.306667 │
└─────────┴──────────────┴──────────┘
